In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression 
from sklearn.preprocessing import StandardScaler


out_path = "../data/results/"

#Feature tables from 03_features
qualis = pd.read_csv(out_path + 'quali_clean.csv')
races = pd.read_csv(out_path + 'races_clean.csv')
qualis.shape, races.shape


((2279, 30), (2278, 36))

In [3]:
# LogGrid: log of grid position, so P1 vs P2 counts for more than P15 vs P16.
# New column so 'Grid' stays raw (the baselines need it, and re-running this cell is safe).
# Chosen on 2025 (Sat 21 Nov): LogGrid alone beat raw Grid, isPole, and both together.
races['LogGrid'] = np.log(races['Grid'])

FEATURES = ['LogGrid', 'DriverForm', 'TeamForm', 'ConstructorPoints', 'DriverShare',
            'Reliability', 'PercentageDifference', 'GapToTeammate']

# Target as a 1-D Series ('Podium', not ['Podium']) so sklearn doesn't warn about a column vector
X_train = races.loc[races['Year'] <= 2024, FEATURES]
Y_train = races.loc[races['Year'] <= 2024, 'Podium']

X_valid = races.loc[races['Year'] == 2025, FEATURES]
Y_valid = races.loc[races['Year'] == 2025, 'Podium']

X_test = races.loc[races['Year'] == 2026, FEATURES]
Y_test = races.loc[races['Year'] == 2026, 'Podium']

races[['Grid', 'LogGrid']].head()


,Grid,LogGrid
0,1.0,0.000000
1,3.0,1.098612
2,5.0,1.609438
3,9.0,2.197225
4,7.0,1.945910


In [4]:
scaler = StandardScaler().fit(X_train) #Sets the mean and sd for the Z-transform

X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

In [10]:
model = LogisticRegression().fit(X_train_s, Y_train)
Y_train.shape
model.predict_proba(X_train_s)[:,1]


array([0.5788593 , 0.16476075, 0.07546071, ..., 0.01262759, 0.00177382,
       0.05918072], shape=(1359,))

In [6]:

res = races.loc[races['Year']==2025].copy()
res['p'] = model.predict_proba(X_valid_s)[:, 1]

res['sum_p'] = res.groupby('Round')['p'].transform('sum')
res['p'] = 3*res['p']/res['sum_p']

In [7]:
# method='first' breaks ties so every race has exactly one rank 1 (same as the 2026 cell)
res['rank'] = res.groupby('Round')['p'].rank(ascending=False, method='first')

res[res['rank'] <= 3] 

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,TeamForm,ConstructorPoints,DriverShare,Reliability,PercentageDifference,GapToTeammate,LogGrid,p,sum_p,rank
1359,4,L NORRIS,NOR,norris,McLaren,FF8000,mclaren,Lando,Norris,Lando Norris,...,11.4,0.0,0.500000,0.909091,0.000000,-0.111732,0.000000,1.131536,1.533734,1.0
1360,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671C6,red_bull,Max,Verstappen,Max Verstappen,...,11.4,0.0,0.500000,0.909091,0.512677,-2.092251,1.098612,0.351699,1.533734,3.0
1367,81,O PIASTRI,PIA,piastri,McLaren,FF8000,mclaren,Oscar,Piastri,Oscar Piastri,...,11.4,0.0,0.500000,0.909091,0.111857,0.111857,0.693147,0.561087,1.533734,2.0
1379,81,O PIASTRI,PIA,piastri,McLaren,FF8000,mclaren,Oscar,Piastri,Oscar Piastri,...,5.0,35.0,0.257143,1.000000,0.000000,-0.160816,0.000000,0.826898,3.137374,1.0
1380,4,L NORRIS,NOR,norris,McLaren,FF8000,mclaren,Lando,Norris,Lando Norris,...,5.0,35.0,0.742857,1.000000,0.161075,0.161075,1.098612,0.555101,3.137374,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1799,81,O PIASTRI,PIA,piastri,McLaren,F47600,mclaren,Oscar,Piastri,Oscar Piastri,...,6.5,770.0,0.485714,0.909091,0.000000,-0.135858,0.000000,0.810877,3.252612,1.0
1801,4,L NORRIS,NOR,norris,McLaren,F47600,mclaren,Lando,Norris,Lando Norris,...,6.5,770.0,0.514286,0.954545,0.136042,0.136042,0.693147,0.656767,3.252612,2.0
1818,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,4781D7,red_bull,Max,Verstappen,Max Verstappen,...,6.6,426.0,0.929577,0.956522,0.000000,-0.995978,0.000000,0.859739,3.137778,1.0
1819,81,O PIASTRI,PIA,piastri,McLaren,F47600,mclaren,Oscar,Piastri,Oscar Piastri,...,6.4,800.0,0.490000,0.913043,0.279782,0.035191,1.098612,0.503614,3.137778,3.0


In [8]:
# Score 2025: model vs the grid baseline
top3 = res[res['rank'] <= 3]          # predicted podium: 3 rows per race
winner = res[res['rank'] == 1]        # predicted winner: 1 row per race

# Winner hit rate: share of races where our rank-1 driver finished 1st
model_win = (winner['Position'] == 1).mean()

# Podium hits: per race, how many of our top 3 actually got a podium, then average over races
model_pod = top3.groupby('Round')['Podium'].sum().mean()

# Grid baseline, computed from raw Grid (not LogGrid) rather than typed in
base_win = (res.loc[res['Grid'] == 1, 'Position'] == 1).mean()
base_pod = res[res['Grid'] <= 3].groupby('Round')['Podium'].sum().mean()

print(f"2025 ({res['Round'].nunique()} races)")
print(f"Model:    winner {model_win:.1%}   podium hits {model_pod:.2f} / 3")
print(f"Baseline: winner {base_win:.1%}   podium hits {base_pod:.2f} / 3")

2025 (24 races)
Model:    winner 62.5%   podium hits 2.33 / 3
Baseline: winner 66.7%   podium hits 2.25 / 3


In [9]:
# Score 2026 so far: model vs the grid baseline, then rate it
res26 = races.loc[races['Year'] == 2026].copy()      # .copy() avoids SettingWithCopyWarning
res26['p'] = model.predict_proba(X_test_s)[:, 1]

# Normalise so each race's probabilities add up to 3 (same as 2025; ranking unchanged)
res26['p'] = 3 * res26['p'] / res26.groupby('Round')['p'].transform('sum')
res26['rank'] = res26.groupby('Round')['p'].rank(ascending=False, method='first')

n_races = res26['Round'].nunique()

# Model
model_win26 = (res26.loc[res26['rank'] == 1, 'Position'] == 1).mean()
model_pod26 = res26[res26['rank'] <= 3].groupby('Round')['Podium'].sum().mean()

# Grid baseline, computed from the data rather than typed in
base_win26 = (res26.loc[res26['Grid'] == 1, 'Position'] == 1).mean()
base_pod26 = res26[res26['Grid'] <= 3].groupby('Round')['Podium'].sum().mean()

print(f"2026 so far ({n_races} races)")
print(f"Model:    winner {model_win26:.1%}   podium hits {model_pod26:.2f} / 3")
print(f"Baseline: winner {base_win26:.1%}   podium hits {base_pod26:.2f} / 3")

# Rating: one race is the smallest change a season can show,
# so a lead smaller than one race's worth counts as a tie (noise)
one_race = 1 / n_races

def rate(lead, unit):
    if lead >= one_race:
        return f"beats the grid ({lead:+.2f} {unit})"
    if lead <= -one_race:
        return f"behind the grid ({lead:+.2f} {unit})"
    return f"level with the grid ({lead:+.2f} {unit}, less than one race of difference)"

print("\nRating")
print("Winner:", rate(model_win26 - base_win26, "hit rate"))
print("Podium:", rate(model_pod26 - base_pod26, "podiums per race"))

2026 so far (20 races)
Model:    winner 70.0%   podium hits 1.90 / 3
Baseline: winner 70.0%   podium hits 1.80 / 3

Rating
Winner: level with the grid (+0.00 hit rate, less than one race of difference)
Podium: beats the grid (+0.10 podiums per race)
